In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment_config


In [0]:
%run ../00-common/02.bronze-helper

In [0]:
source_file = f'{landing_folder_path}/{v_batch_id}/circuits.csv'
table_name = f"{catalog_name}.{bronze_schema}.circuits"
display(source_file)
display(table_name)

#Ingest circuits csv file
1. Read the file using spark dataframe reader API
2. Add Metadata Columns
-        Source File
-        Ingestion Timestamp
3. Write to bronze delta table

##Step 1 Read the file using spark dataframe reader API

In [0]:
from pyspark.sql.types import StructType, StructField,StringType,DoubleType
circuits_schema = StructType([
    StructField('circuitId',  StringType()),
    StructField('url',        StringType()),
    StructField('circuitName',StringType()),
    StructField('lat',        DoubleType()),
    StructField('long',       DoubleType()),
    StructField('locality',   StringType()),
    StructField('country',    StringType())
   
])

In [0]:
circuits_df = (
    spark.read
    .format('csv')
    .option('header','true')
    .option('mode','FAILFAST')
    .schema(circuits_schema)
    .load(source_file))
    

In [0]:
display(circuits_df)

##Step2 - Add meta data columns
 - Source File
 - Ingestion Timeline

In [0]:

circuits_final_df = add_ingestion_metadata(circuits_df)


In [0]:
display(circuits_final_df)

In [0]:
write_to_bronze(input_df = circuits_final_df , 
                target_table=table_name,
                batch_id=v_batch_id)
  

### Step 3 - Write to bronze delta table


In [0]:
display(spark.table(table_name))